In [1]:
# Install necessary libraries
!pip install requests pytz azure-servicebus

import requests
import json
import time
from datetime import datetime
import pytz
from azure.servicebus import ServiceBusClient, ServiceBusMessage

StatementMeta(, 27d0b5a0-9ba5-4de6-8a78-75d7084fc7b3, 3, Finished, Available, Finished)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.0/99.0 kB 2.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.5/412.5 kB 15.0 MB/s eta 0:00:00


# Automate data streaming every 2 seconds!

In [2]:
# --- CONFIGURATION ---
# Microsoft Fabric EventStream Connection Details
CONNECTION_STR = ""

# Target API (Binance US Ticker)
API_URL = "https://api.binance.us/api/v3/ticker/price"

def fetch_api_data():
    """
    Fetches real-time price data from the API.
    Returns: A list of data records.
    """
    try:
        response = requests.get(API_URL)
        response.raise_for_status()
        data = response.json()

        # Ensure output is always a list, even if API returns a single dictionary
        return [data] if isinstance(data, dict) else data
    except requests.exceptions.RequestException as e:
        print(f"⚠️ API Error: {e}")
        return None

def add_metadata(data):
    """
    Enriches the raw API data with required schema fields including 
    UTC and Local (Helsinki) timestamps.
    """
    # Define timezones
    helsinki_tz = pytz.timezone("Europe/Helsinki")
    utc_tz = pytz.utc

    # Get current times
    now_helsinki = datetime.now(helsinki_tz)
    now_utc = datetime.now(utc_tz)

    # Format timestamps according to requested JSON schema
    iso_utc = now_utc.strftime("%Y-%m-%dT%H:%M:%SZ")
    iso_local = now_helsinki.strftime("%Y-%m-%dT%H:%M:%S")

    for record in data:
        record["source"] = "binance_us"
        record["event_time_utc"] = iso_utc
        record["event_time_local"] = iso_local
        # 'ingestion_time' reflects exactly when the script processed the record
        record["ingestion_time"] = datetime.now(utc_tz).isoformat()
        
    return data

def send_to_fabric(messages, connection_string):
    """
    Parses the connection string and sends the list of records to 
    the Azure Service Bus (Fabric EventStream).
    """
    # Extract EntityPath (Queue/Topic name) from the connection string
    entity_path = next((p.split('=')[1] for p in connection_string.split(';') if p.startswith('EntityPath=')), None)

    if not entity_path:
        raise ValueError("Invalid Connection String: EntityPath missing.")

    client = ServiceBusClient.from_connection_string(connection_string)
    try:
        with client.get_queue_sender(entity_path) as sender:
            # Batch the messages for efficient transmission
            batch_message = [ServiceBusMessage(json.dumps(msg)) for msg in messages]
            sender.send_messages(batch_message)
            print(f"🚀 Successfully sent {len(messages)} records.")
    except Exception as e:
        print(f"❌ Transmission Error: {e}")
    finally:
        client.close()

# --- MAIN EXECUTION LOOP ---
print(f"Starting Real-Time Stream: {API_URL} -> Microsoft Fabric...")

while True:
    raw_payload = fetch_api_data()
    
    if raw_payload:
        # 1. Transform: Add the required timestamps and source tags
        enriched_data = add_metadata(raw_payload)
        
        # 2. Load: Push the data to Fabric
        send_to_fabric(enriched_data, CONNECTION_STR)
        
        # Log a snippet of the first record for verification
        sample = enriched_data[0]
        print(f"   [Sample] {sample.get('symbol')}: {sample.get('price')} | Time: {sample.get('event_time_local')}")
    
    # Wait 2 seconds before the next pull
    time.sleep(2)

StatementMeta(, 27d0b5a0-9ba5-4de6-8a78-75d7084fc7b3, 4, Finished, Cancelled, Cancelled)

Starting Real-Time Stream: https://api.binance.us/api/v3/ticker/price -> Microsoft Fabric...
🚀 Successfully sent 615 records.
   [Sample] BTCUSD4: 22882.5400 | Time: 2026-01-27T23:10:58
🚀 Successfully sent 615 records.
   [Sample] BTCUSD4: 22882.5400 | Time: 2026-01-27T23:11:01
🚀 Successfully sent 615 records.
   [Sample] BTCUSD4: 22882.5400 | Time: 2026-01-27T23:11:05
🚀 Successfully sent 615 records.
   [Sample] BTCUSD4: 22882.5400 | Time: 2026-01-27T23:11:08
🚀 Successfully sent 615 records.
   [Sample] BTCUSD4: 22882.5400 | Time: 2026-01-27T23:11:11
